<a href="https://colab.research.google.com/github/SolKacil/matematicas-para-ia/blob/main/python/01-algebra-lineal/04_autovalores_y_autovectores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 04 &middot; Autovalores y autovectores

**Módulo 1 — Álgebra lineal y geometría diferencial**

Este notebook acompaña al documento `Matematicas_para_IA_04.pdf`, basado en Deisenroth, Faisal y
Ong (2020), *Mathematics for Machine Learning*, secciones 4.1–4.4.

Al multiplicar una matriz $A$ por un vector arbitrario $\mathbf{x}$, el resultado $A\mathbf{x}$
apunta por lo general en una dirección distinta de la de $\mathbf{x}$. Existen, sin embargo,
direcciones para las que la matriz se limita a estirar o contraer el vector sin desviarlo. Esas
direcciones son los **autovectores**, y el factor correspondiente el **autovalor**. Identificarlas
equivale a encontrar el sistema de coordenadas en el que la transformación es más simple posible:
una escala independiente por eje.

Este notebook cierra el módulo porque emplea todo lo anterior. El polinomio característico se define
con un **determinante** (notebook 03); los autovectores se calculan resolviendo un sistema
**homogéneo**, es decir, un espacio nulo (notebook 02); la ortogonalidad de los autovectores de una
matriz simétrica se verifica con el **producto punto** (notebook 03); y la diagonalización se
escribe con productos e inversas de matrices (notebooks 00 y 02).

## Al terminar será posible

- Verificar la definición $A\mathbf{v} = \lambda\mathbf{v}$ y distinguir un autovector de un vector
  cualquiera.
- Obtener el **polinomio característico** $\det(A - \lambda I) = 0$ y resolverlo, tanto de forma
  simbólica como numérica.
- Calcular los **autovectores** de cada autovalor resolviendo $(A - \lambda I)\mathbf{v} = \mathbf{0}$
  con el método del $-1$ del notebook 02.
- Enunciar y comprobar las propiedades espectrales de las matrices **simétricas** y **ortogonales**.
- **Diagonalizar** una matriz, $A = PDP^{-1}$, y emplear la factorización para calcular potencias.
- Implementar el **método de la potencia** y relacionarlo con PCA y con la estabilidad de sistemas
  dinámicos.

## Qué se da por sabido

Los notebooks [02 · Sistemas lineales](02_sistemas_lineales_y_espacio_nulo.ipynb) —espacio nulo y
método del $-1$— y [03 · Determinante, norma y producto punto](03_determinante_norma_producto_punto.ipynb)
—determinante, ortogonalidad y matrices ortogonales—.

## Cómo usar este notebook

1. Con el botón **Open in Colab** no se requiere instalación alguna.
2. Las celdas se ejecutan en orden con `Shift + Enter`.
3. Conviene comprobar en cada caso la identidad $A\mathbf{v} = \lambda\mathbf{v}$ directamente: es
   la definición, y verificarla cuesta una línea de código.
4. La sección final, **Tu turno**, contiene los ejercicios propuestos del documento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from fractions import Fraction

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5.5, 5.5)

print("numpy:", np.__version__, " sympy:", sp.__version__)

---

## 1. Definición

Sea $A \in \mathbb{R}^{n \times n}$. Se dice que $\lambda \in \mathbb{R}$ es un **autovalor** de $A$
y que $\mathbf{v} \in \mathbb{R}^n \setminus \{\mathbf{0}\}$ es su **autovector** asociado si

$$A\mathbf{v} = \lambda\mathbf{v}.$$

Multiplicar $\mathbf{v}$ por la matriz produce el mismo resultado que multiplicarlo por el número
$\lambda$. Se excluye $\mathbf{v} = \mathbf{0}$ porque la igualdad se cumpliría para cualquier $A$ y
cualquier $\lambda$, y la definición perdería contenido.

Con $A = \begin{bmatrix} 4 & 1 \\ 2 & 3 \end{bmatrix}$, el vector $\mathbf{v} = (1,1)$ es autovector
con autovalor $5$, mientras que $\mathbf{w} = (1,0)$ no es autovector: $A\mathbf{w} = (4,2)$ no es
múltiplo de $\mathbf{w}$.

In [ ]:
A = np.array([[4.0, 1.0],
              [2.0, 3.0]])

v = np.array([1.0, 1.0])
w = np.array([1.0, 0.0])

print("A @ v =", A @ v, "   5 * v =", 5 * v, "  -> v SI es autovector, con lambda = 5")
print("A @ w =", A @ w, "   no es multiplo de w =", w, " -> w NO es autovector")

# Cualquier multiplo de un autovector tambien lo es, con el mismo autovalor.
for c in [2.0, -3.0, 0.5]:
    print(f"A @ ({c} * v) = {A @ (c * v)}   =  5 * ({c} * v) = {5 * (c * v)}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 5.5))

for ax, (u, nombre, color) in zip(ejes, [(v, "v", "tab:blue"), (w, "w", "tab:red")]):
    Au = A @ u
    ax.quiver(0, 0, u[0], u[1], angles="xy", scale_units="xy", scale=1, color=color, width=0.013)
    ax.quiver(0, 0, Au[0], Au[1], angles="xy", scale_units="xy", scale=1,
              color=color, alpha=0.45, width=0.013)
    ax.annotate(f"{nombre} = ({u[0]:g}, {u[1]:g})", u, textcoords="offset points",
                xytext=(8, -12), color=color)
    ax.annotate(f"A{nombre} = ({Au[0]:g}, {Au[1]:g})", Au, textcoords="offset points",
                xytext=(6, 6), color=color)
    recta = np.linspace(-1, 6, 10)
    if u[1] != 0 or u[0] != 0:
        ax.plot(recta * u[0], recta * u[1], "--", color="gray", lw=1)
    ax.set_xlim(-1, 6); ax.set_ylim(-1, 6); ax.set_aspect("equal")
    ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3)

ejes[0].set_title("v es autovector: Av cae sobre la misma recta", fontsize=10)
ejes[1].set_title("w no lo es: Aw se sale de la recta", fontsize=10)
plt.show()

---

## 2. El polinomio característico

Partiendo de la definición y agrupando términos:

$$A\mathbf{v} = \lambda\mathbf{v} \;\Longrightarrow\; A\mathbf{v} - \lambda\mathbf{v} = \mathbf{0}
\;\Longrightarrow\; (A - \lambda I)\mathbf{v} = \mathbf{0}.$$

La identidad $I$ aparece porque $\lambda$ es un número y $A$ una matriz: no se pueden restar
directamente, pero $\lambda I$ sí es una matriz del mismo tamaño que $A$.

Lo que se busca es un vector $\mathbf{v} \neq \mathbf{0}$ que resuelva ese sistema **homogéneo**. Por
lo visto en el notebook 02, un sistema homogéneo tiene soluciones distintas de cero exactamente
cuando la matriz es singular, y por el notebook 03 eso equivale a que su determinante se anule. Ésa
es la condición que define los autovalores:

$$\boxed{\det(A - \lambda I) = 0}$$

La expresión de la izquierda es un polinomio en $\lambda$, llamado **polinomio característico**, y
sus raíces son los autovalores de $A$.

In [ ]:
lam = sp.symbols("lambda")
A_sp = sp.Matrix([[4, 1],
                  [2, 3]])

caracteristico = (A_sp - lam * sp.eye(2)).det()
print("A - lambda I =")
sp.pprint(A_sp - lam * sp.eye(2))
print("\npolinomio caracteristico:", sp.expand(caracteristico))
print("factorizado:", sp.factor(caracteristico))
print("raices (autovalores):", sp.solve(sp.Eq(caracteristico, 0), lam))

---

## 3. Ejemplo completo, paso a paso

Para $A = \begin{bmatrix} 4 & 1 \\ 2 & 3 \end{bmatrix}$:

**Paso 1.** $A - \lambda I = \begin{bmatrix} 4-\lambda & 1 \\ 2 & 3-\lambda \end{bmatrix}$.

**Paso 2.** $\det(A - \lambda I) = (4-\lambda)(3-\lambda) - 2 = \lambda^2 - 7\lambda + 10$, y de
$(\lambda - 5)(\lambda - 2) = 0$ se obtienen $\lambda = 5$ y $\lambda = 2$.

**Paso 3.** Para cada autovalor se resuelve $(A - \lambda I)\mathbf{v} = \mathbf{0}$, que es
exactamente el cálculo de un espacio nulo.

- Con $\lambda = 5$: $\begin{bmatrix} -1 & 1 \\ 2 & -2 \end{bmatrix}\mathbf{v} = \mathbf{0}
  \Rightarrow v_1 = v_2 \Rightarrow \mathbf{v} = (1,1)$.
- Con $\lambda = 2$: $\begin{bmatrix} 2 & 1 \\ 2 & 1 \end{bmatrix}\mathbf{v} = \mathbf{0}
  \Rightarrow v_2 = -2v_1 \Rightarrow \mathbf{v} = (1,-2)$.

Cada autovector representa una **dirección completa** —una recta por el origen— y no un vector
único: cualquier múltiplo no nulo es también autovector con el mismo autovalor. Al conjunto de todos
esos múltiplos, junto con $\mathbf{0}$, se le llama **espacio propio**, y es de nuevo un subespacio
vectorial (notebook 01): es el espacio nulo de $A - \lambda I$.

Las funciones siguientes reutilizan el método del $-1$ del notebook 02 para obtener los autovectores
en aritmética exacta.

In [ ]:
def rref(M):
    """Forma escalonada reducida en aritmetica racional exacta (notebook 02)."""
    R = [[Fraction(x) for x in fila] for fila in M]
    m, n = len(R), len(R[0])
    pivotes, fila = [], 0
    for col in range(n):
        candidatos = [i for i in range(fila, m) if R[i][col] != 0]
        if not candidatos:
            continue
        i = min(candidatos, key=lambda k: (abs(R[k][col]) != 1, abs(R[k][col])))
        R[fila], R[i] = R[i], R[fila]
        p = R[fila][col]
        if p != 1:
            R[fila] = [x / p for x in R[fila]]
        for k in range(m):
            if k != fila and R[k][col] != 0:
                f = R[k][col]
                R[k] = [a - f * c for a, c in zip(R[k], R[fila])]
        pivotes.append(col)
        fila += 1
        if fila == m:
            break
    return R, pivotes


def nucleo_menos_uno(M):
    """Base del espacio nulo por el metodo del -1 (notebook 02)."""
    R, pivotes = rref(M)
    n = len(M[0])
    libres = [j for j in range(n) if j not in pivotes]
    At = [[Fraction(0)] * n for _ in range(n)]
    for i, j in enumerate(pivotes):
        At[j] = R[i][:]
    for j in libres:
        f = [Fraction(0)] * n
        f[j] = Fraction(-1)
        At[j] = f
    return [[At[i][j] for i in range(n)] for j in libres]


def espacio_propio(A, autovalor):
    """Base del espacio propio de un autovalor: el nucleo de A - lambda I."""
    n = len(A)
    desplazada = [[Fraction(A[i][j]) - (Fraction(autovalor) if i == j else 0)
                   for j in range(n)] for i in range(n)]
    return nucleo_menos_uno(desplazada)


def representante(vec):
    """Reescala un autovector para que su primera entrada no nula valga 1.

    Cualquier multiplo no nulo es igualmente valido; esto solo elige uno
    canonico, para poder compararlo con el del documento.
    """
    primera = next(x for x in vec if x != 0)
    return [x / primera for x in vec]


A_exacta = [[4, 1],
            [2, 3]]

for lam_val in [5, 2]:
    for vec in espacio_propio(A_exacta, lam_val):
        canonico = representante(vec)
        vf = np.array([float(x) for x in canonico])
        print(f"lambda = {lam_val}:  v = ({', '.join(str(x) for x in canonico)})"
              f"   A @ v = {np.array(A_exacta, float) @ vf}"
              f"   {lam_val} * v = {lam_val * vf}")
        print(f"            tal como lo devuelve el metodo del -1: "
              f"({', '.join(str(x) for x in vec)})  <- el mismo, reescalado")

In [ ]:
# La rutina numerica de NumPy devuelve lo mismo, con los autovectores normalizados.
valores, vectores = np.linalg.eig(A)

print("autovalores:", valores)
print("autovectores (en columnas):\n", vectores)

for k in range(len(valores)):
    lam_k, v_k = valores[k], vectores[:, k]
    print(f"\nlambda = {lam_k:.4f}")
    print(f"  v          = {v_k}   (norma {np.linalg.norm(v_k):.4f})")
    print(f"  A @ v      = {A @ v_k}")
    print(f"  lambda * v = {lam_k * v_k}")
    print(f"  coinciden  : {np.allclose(A @ v_k, lam_k * v_k)}")

# Los autovectores exactos y los de NumPy generan las mismas rectas.
print("\nv exacto (1, 1) normalizado:", np.array([1.0, 1.0]) / np.sqrt(2))

---

## 4. Matrices simétricas

Con $A = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$, que es simétrica, el mismo procedimiento da
$\det(A - \lambda I) = (2-\lambda)^2 - 1 = 0$, de donde $\lambda = 3$ y $\lambda = 1$, con
autovectores $\mathbf{v}_1 = (1,1)$ y $\mathbf{v}_2 = (1,-1)$. Su producto punto es

$$\mathbf{v}_1 \cdot \mathbf{v}_2 = (1)(1) + (1)(-1) = 0,$$

de modo que son **ortogonales**. No es una coincidencia del ejemplo.

**Propiedades de las matrices simétricas ($A^\top = A$).**

- Todos sus autovalores son **reales**, nunca complejos.
- Los autovectores asociados a autovalores distintos son siempre **ortogonales** entre sí.

Este resultado es el **teorema espectral**, y es la razón por la que las matrices simétricas son
tratables: toda matriz de covarianza y toda matriz de Gram $X^\top X$ (notebook 00) lo son.

In [ ]:
S = np.array([[2.0, 1.0],
              [1.0, 2.0]])

for lam_val in [3, 1]:
    for vec in espacio_propio([[2, 1], [1, 2]], lam_val):
        canonico = representante(vec)
        print(f"lambda = {lam_val}:  v = ({', '.join(str(x) for x in canonico)})")

v1 = np.array([1.0, 1.0])
v2 = np.array([1.0, -1.0])
print("\nv1 . v2 =", v1 @ v2, " -> ortogonales")

valores_s, vectores_s = np.linalg.eigh(S)      # eigh aprovecha la simetria
print("\nautovalores (eigh):", valores_s)
print("autovectores:\n", vectores_s)
print("ortonormales: V.T @ V =\n", vectores_s.T @ vectores_s)

In [ ]:
# El teorema espectral, comprobado sobre matrices simetricas al azar.
rng = np.random.default_rng(0)
for n in [3, 5, 8]:
    X = rng.normal(size=(n, n))
    M = X + X.T                                 # simetrica por construccion
    valores = np.linalg.eigvals(M)
    _, V = np.linalg.eigh(M)
    print(f"n = {n}: autovalores reales: {np.allclose(valores.imag, 0)}   "
          f"autovectores ortonormales: {np.allclose(V.T @ V, np.eye(n))}")

# Una matriz no simetrica puede tener autovalores complejos.
R = np.array([[0.0, -1.0],
              [1.0, 0.0]])                      # rotacion de 90 grados
print("\nautovalores de la rotacion de 90 grados:", np.linalg.eigvals(R))

### 4.1 Matrices ortogonales

Para una matriz ortogonal ($A^\top A = I$, notebook 03) todos los autovalores cumplen
$|\lambda| = 1$; pueden ser complejos, pero su módulo siempre vale uno.

La razón es geométrica: una matriz ortogonal representa una rotación o una reflexión, y una rotación
distinta de $0°$ o $180°$ no deja fija ninguna dirección real —gira todos los vectores—, de modo que
en general no tiene autovectores reales. Los autovalores de una rotación de ángulo $\theta$ son
$e^{\pm i\theta}$, cuyo módulo es $1$, y esa fase es exactamente el ángulo girado.

In [ ]:
for grados in [30.0, 90.0, 180.0]:
    t = np.radians(grados)
    Rot = np.array([[np.cos(t), -np.sin(t)],
                    [np.sin(t), np.cos(t)]])
    valores = np.linalg.eigvals(Rot)
    print(f"rotacion de {grados:>5.1f} grados: autovalores = {np.round(valores, 4)}   "
          f"modulos = {np.round(np.abs(valores), 6)}")

# Una reflexion si tiene autovectores reales: el eje de reflexion y su perpendicular.
Refl = np.array([[1.0, 0.0],
                 [0.0, -1.0]])
valores, vectores = np.linalg.eig(Refl)
print(f"\nreflexion: autovalores = {valores}  (modulos {np.abs(valores)})")
print("autovectores:\n", vectores, "\n<- el eje que se conserva y el que se invierte")

---

## 5. Diagonalización

Si se colocan los autovectores de $A$ como columnas de una matriz $P$ y los autovalores
correspondientes en la diagonal de una matriz $D$ —en el mismo orden—, entonces

$$A = PDP^{-1}.$$

Para la matriz de la sección 3, $P = \begin{bmatrix} 1 & 1 \\ 1 & -2 \end{bmatrix}$ y
$D = \begin{bmatrix} 5 & 0 \\ 0 & 2 \end{bmatrix}$.

La factorización expresa la transformación como tres pasos: $P^{-1}$ cambia a las coordenadas de los
autovectores, $D$ escala cada eje de manera independiente y $P$ regresa a las coordenadas
originales. En el sistema adecuado, la matriz se reduce a una lista de factores de escala.

**El caso simétrico.** Si $A$ es simétrica, sus autovectores son ortogonales; normalizándolos, $P$
se convierte en una matriz ortogonal y, por lo visto en el notebook 03, $P^{-1} = P^\top$:

$$A = PDP^\top .$$

No hace falta invertir nada, sólo transponer.

In [ ]:
P = np.array([[1.0, 1.0],
              [1.0, -2.0]])
D = np.diag([5.0, 2.0])

print("P D P^-1 =\n", P @ D @ np.linalg.inv(P))
print("\nA =\n", A)
print("\ncoinciden:", np.allclose(P @ D @ np.linalg.inv(P), A))

# Caso simetrico: P es ortogonal y basta transponer.
valores_s, P_s = np.linalg.eigh(S)
D_s = np.diag(valores_s)
print("\nS simetrica:  P ortogonal:", np.allclose(P_s.T @ P_s, np.eye(2)))
print("P D P.T =\n", P_s @ D_s @ P_s.T)
print("coincide con S:", np.allclose(P_s @ D_s @ P_s.T, S))

### 5.1 Para qué sirve diagonalizar

El uso inmediato es el cálculo de **potencias**. Como

$$A^k = (PDP^{-1})(PDP^{-1})\cdots(PDP^{-1}) = PD^kP^{-1},$$

los factores intermedios $P^{-1}P$ se cancelan y sólo queda elevar la matriz **diagonal**, lo que
consiste en elevar cada número de la diagonal. El costo pasa de $k$ productos de matrices a uno solo
más $n$ potencias escalares.

Esto tiene una consecuencia que se retoma en la sección 6: el comportamiento de $A^k$ cuando $k$
crece está determinado por completo por los autovalores. Si todos cumplen $|\lambda| < 1$, entonces
$A^k \to 0$; si alguno tiene $|\lambda| > 1$, las entradas de $A^k$ crecen sin límite.

In [ ]:
def potencia_por_diagonalizacion(P, D, k):
    return P @ np.diag(np.diag(D) ** k) @ np.linalg.inv(P)


k = 12
directa = np.linalg.matrix_power(A, k)
diagonal = potencia_por_diagonalizacion(P, D, k)

print(f"A^{k} directa =\n", directa)
print(f"\nA^{k} = P D^{k} P^-1 =\n", diagonal)
print("\ncoinciden:", np.allclose(directa, diagonal))
print(f"\nD^{k} = diag({np.diag(D)} ** {k}) = {np.diag(D) ** k}")

In [ ]:
# El comportamiento a largo plazo depende solo del modulo de los autovalores.
casos = {
    "todos |lambda| < 1": np.array([[0.5, 0.1], [0.0, 0.3]]),
    "algun |lambda| > 1": np.array([[1.2, 0.1], [0.0, 0.9]]),
    "el mayor |lambda| = 1": np.array([[1.0, 0.0], [0.0, 0.5]]),
}

for nombre, M in casos.items():
    radio = max(abs(np.linalg.eigvals(M)))
    normas = [np.linalg.norm(np.linalg.matrix_power(M, k)) for k in [1, 10, 50]]
    print(f"{nombre:>22}: radio espectral = {radio:.3f}   "
          f"||M^k|| para k=1,10,50: {np.round(normas, 4)}")

---

## 6. Lectura en aprendizaje automático

### 6.1 Análisis de componentes principales

PCA calcula los autovectores y autovalores de la **matriz de covarianza** de los datos. Los
autovectores señalan las direcciones de máxima variación y los autovalores indican cuánta varianza
hay en cada una. Como la matriz de covarianza es simétrica por construcción, el teorema espectral
garantiza que sus autovectores son ortogonales entre sí: las componentes principales no comparten
información, que es precisamente lo que se busca de un sistema de coordenadas para describir los
datos.

In [ ]:
# Datos con una direccion dominante clara.
rng = np.random.default_rng(3)
base = rng.normal(size=(300, 2)) @ np.array([[2.5, 0.0], [0.0, 0.6]])
angulo = np.radians(30)
giro = np.array([[np.cos(angulo), -np.sin(angulo)],
                 [np.sin(angulo), np.cos(angulo)]])
datos = base @ giro.T + np.array([1.0, -0.5])

centrados = datos - datos.mean(axis=0)
covarianza = (centrados.T @ centrados) / (len(centrados) - 1)

valores_pca, vectores_pca = np.linalg.eigh(covarianza)
orden = np.argsort(valores_pca)[::-1]                 # de mayor a menor varianza
valores_pca, vectores_pca = valores_pca[orden], vectores_pca[:, orden]

print("matriz de covarianza (simetrica):\n", covarianza)
print("\nautovalores (varianza por direccion):", np.round(valores_pca, 4))
print("autovectores (componentes principales), en columnas:\n", np.round(vectores_pca, 4))
print("\nortogonales:", np.isclose(vectores_pca[:, 0] @ vectores_pca[:, 1], 0.0))
print(f"varianza explicada por la primera componente: {valores_pca[0] / valores_pca.sum():.1%}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(centrados[:, 0], centrados[:, 1], s=10, alpha=0.35, label="datos centrados")
colores = ["tab:red", "tab:green"]
for k in range(2):
    direccion = vectores_pca[:, k] * np.sqrt(valores_pca[k]) * 2.5
    ax.quiver(0, 0, direccion[0], direccion[1], angles="xy", scale_units="xy", scale=1,
              color=colores[k], width=0.012,
              label=f"componente {k + 1}  (lambda = {valores_pca[k]:.2f})")
ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
ax.legend(fontsize=9); ax.set_title("Los autovectores de la covarianza son las componentes principales")
plt.show()

### 6.2 Estabilidad de sistemas dinámicos

En un sistema que evoluciona según $\mathbf{x}_{t+1} = A\mathbf{x}_t$ —por ejemplo el estado interno
de una red recurrente en cada paso temporal— el estado en el instante $t$ es $A^t\mathbf{x}_0$, y por
la sección 5 su comportamiento depende únicamente de los autovalores de $A$. Si todos cumplen
$|\lambda| < 1$ el sistema converge a cero; si alguno cumple $|\lambda| > 1$, la magnitud crece sin
límite.

Ésta es la formulación precisa del problema de los **gradientes que se desvanecen o explotan** en
redes recurrentes: la misma matriz se aplica una vez por paso, de modo que el radio espectral queda
elevado a la longitud de la secuencia. El caso $|\lambda| = 1$ —al que corresponden las matrices
ortogonales del notebook 03— es la frontera exacta en la que la señal ni crece ni decae, y es el
motivo de la inicialización ortogonal en ese tipo de redes.

In [ ]:
rng = np.random.default_rng(1)
x0 = rng.normal(size=2)
pasos = 40

fig, ax = plt.subplots(figsize=(7, 4.2))
for nombre, M, color in [("radio 0.85 (estable)", np.array([[0.8, 0.2], [0.0, 0.85]]), "tab:blue"),
                         ("radio 1.00 (frontera)", np.array([[0.0, -1.0], [1.0, 0.0]]), "tab:green"),
                         ("radio 1.15 (inestable)", np.array([[1.1, 0.2], [0.0, 1.15]]), "tab:red")]:
    x = x0.copy()
    normas = [np.linalg.norm(x)]
    for _ in range(pasos):
        x = M @ x
        normas.append(np.linalg.norm(x))
    radio = max(abs(np.linalg.eigvals(M)))
    ax.semilogy(normas, color=color, label=f"{nombre}: radio = {radio:.2f}")
    print(f"{nombre:>24}: ||x_0|| = {normas[0]:.3f} -> ||x_{pasos}|| = {normas[-1]:.3e}")

ax.set_xlabel("paso t"); ax.set_ylabel(r"$\|x_t\|$  (escala logaritmica)")
ax.set_title("El radio espectral decide el comportamiento a largo plazo")
ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.show()

### 6.3 El método de la potencia

Para matrices grandes, resolver el polinomio característico es inviable: no existe fórmula por
radicales para grado mayor que cuatro, y el problema está mal condicionado. Los algoritmos reales son
iterativos.

El **método de la potencia** obtiene el autovalor dominante —el de mayor módulo— partiendo de un
vector arbitrario $\mathbf{x}_0$ y calculando repetidamente $\mathbf{x}_{k+1} = A\mathbf{x}_k$. La
razón de que funcione se ve al escribir $\mathbf{x}_0$ en la base de autovectores: cada componente
queda multiplicada por $\lambda_i^k$, de modo que la asociada al mayor $|\lambda|$ domina a las
demás de forma exponencial y la dirección de $\mathbf{x}_k$ converge a la del autovector dominante.

La tabla siguiente reproduce la del documento, con $A = \begin{bmatrix} 4 & 1 \\ 2 & 3 \end{bmatrix}$
y $\mathbf{x}_0 = (1,0)$: la segunda coordenada del vector normalizado se aproxima a $1$, es decir, a
la dirección $(1,1)$ que ya se había calculado analíticamente.

Este método y sus variantes están en la base de algoritmos como PageRank, que busca el autovector
dominante de una matriz que describe los enlaces entre páginas web.

In [ ]:
def metodo_potencia(A, x0, iteraciones=10):
    """Itera x <- A x y devuelve la sucesion de vectores y el cociente de Rayleigh."""
    x = np.asarray(x0, dtype=float)
    historial = []
    for k in range(1, iteraciones + 1):
        x = A @ x
        normalizado = x / np.abs(x).max()               # entrada mayor igual a 1
        rayleigh = (x @ (A @ x)) / (x @ x)              # estimacion del autovalor
        historial.append((k, x.copy(), normalizado.copy(), rayleigh))
    return historial


print(f"{'iter':>4} | {'x_k':>22} | {'normalizado':>18} | {'estimacion de lambda':>20}")
print("-" * 76)
for k, xk, norm, ray in metodo_potencia(A, [1.0, 0.0], 8):
    print(f"{k:>4} | {str(np.round(xk, 1)):>22} | {str(np.round(norm, 3)):>18} | {ray:>20.6f}")

print("\nautovalor dominante exacto: 5.0   autovector dominante: (1, 1)")

In [ ]:
# En una matriz grande el metodo converge sin calcular ningun determinante.
rng = np.random.default_rng(5)
n = 200
G = rng.normal(size=(n, n))
G = G + G.T                                      # simetrica: autovalores reales

x = rng.normal(size=n)
for _ in range(200):
    x = G @ x
    x = x / np.linalg.norm(x)

estimacion = x @ (G @ x)
exacto = np.linalg.eigvalsh(G)[[0, -1]]
dominante = exacto[np.argmax(np.abs(exacto))]

print(f"estimacion por el metodo de la potencia: {estimacion:.6f}")
print(f"autovalor dominante segun eigvalsh     : {dominante:.6f}")
print(f"error relativo: {abs(estimacion - dominante) / abs(dominante):.2e}")
print(f"\n||G x - lambda x|| = {np.linalg.norm(G @ x - estimacion * x):.2e}  <- x es el autovector")

---

## Tu turno

Los ejercicios son los propuestos en el documento `Matematicas_para_IA_04.pdf`. Sus respuestas están
en el PDF; el objetivo aquí es **calcularlas a mano primero y verificarlas después con código**,
apoyándose en `espacio_propio`, `np.linalg.eig` y `np.linalg.eigh`.

**1.** Encontrar los autovalores y autovectores de $A = \begin{bmatrix} 3 & 0 \\ 0 & 5 \end{bmatrix}$.
Antes de calcular nada, conjeturar el resultado a partir de la forma de la matriz y comprobar
después la conjetura. ¿Qué ocurre en general con una matriz diagonal?

**2.** Encontrar los autovalores de $A = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix}$ y verificar
que sus autovectores son ortogonales. Relacionar el resultado con el teorema espectral.

**3.** Con $A = \begin{bmatrix} 4 & 1 \\ 2 & 3 \end{bmatrix}$ y sus autovalores $\lambda = 5, 2$,
calcular $\det(A)$ como producto de los autovalores y contrastarlo con el cálculo directo.
Comprobar además que la **traza** —la suma de la diagonal— es igual a la suma de los autovalores.

**4.** Obtener el polinomio característico de
$A = \begin{bmatrix} 2 & 1 & 0 \\ 1 & 2 & 1 \\ 0 & 1 & 2 \end{bmatrix}$ con `sympy`, resolverlo, y
calcular los espacios propios con `espacio_propio`. Verificar que los tres autovectores son
ortogonales entre sí.

**5.** Diagonalizar la matriz del ejercicio 2, escribir $A = PDP^\top$ con $P$ ortogonal y usar la
factorización para calcular $A^{10}$. Comparar con `np.linalg.matrix_power`.

**6.** Implementar el método de la potencia con **normalización por la norma euclidiana** en cada
paso, en lugar de por la entrada mayor, y comprobar que converge a la misma dirección. ¿Por qué es
preferible normalizar en cada iteración en lugar de al final?

**7.** Construir una matriz $2 \times 2$ cuyos autovalores sean $0.5$ y $0.9$, partiendo de $P$ y $D$
y formando $PDP^{-1}$. Iterar $\mathbf{x}_{t+1} = A\mathbf{x}_t$ y verificar que la sucesión converge
a cero. Repetir con autovalores $1.1$ y $0.9$ y observar la diferencia.

**8.** Generar una matriz de datos de $200 \times 3$ cuya tercera columna sea casi una combinación
lineal de las dos primeras. Calcular los autovalores de su matriz de covarianza y comprobar que uno
de ellos es mucho menor que los otros. ¿Qué dice eso sobre la dimensión efectiva de los datos?

In [ ]:
# Tu codigo aqui.
A_ej = np.array([[3.0, 0.0],
                 [0.0, 5.0]])

# 1.

---

## Resumen

| Concepto | Definición | En NumPy |
|---|---|---|
| Autovalor y autovector | $A\mathbf{v} = \lambda\mathbf{v}$, $\mathbf{v} \neq \mathbf{0}$ | `np.linalg.eig` |
| Polinomio característico | $\det(A - \lambda I) = 0$ | `sympy` para la forma simbólica |
| Espacio propio | $N(A - \lambda I)$ | `espacio_propio` (exacto) |
| Matriz simétrica | autovalores reales, autovectores ortogonales | `np.linalg.eigh` |
| Diagonalización | $A = PDP^{-1}$; si $A$ es simétrica, $A = PDP^\top$ | `eig` / `eigh` |
| Potencias | $A^k = PD^kP^{-1}$ | `np.linalg.matrix_power` |
| Autovalor dominante | método de la potencia | iterar $\mathbf{x} \leftarrow A\mathbf{x}$ |

| Relación | Enunciado |
|---|---|
| Determinante | $\det(A) = \prod_i \lambda_i$ |
| Traza | $\operatorname{tr}(A) = \sum_i \lambda_i$ |
| Singularidad | $A$ es singular $\iff$ algún $\lambda_i = 0$ |
| Matriz ortogonal | $\lvert\lambda_i\rvert = 1$ para todo $i$ |
| Estabilidad | $A^k \to 0 \iff$ todos los $\lvert\lambda_i\rvert < 1$ |

Cuatro ideas para retener:

1. **Un autovector es una dirección que la matriz no desvía.** Encontrar los autovectores equivale a
   encontrar el sistema de coordenadas donde la transformación es una escala por eje.
2. **El cálculo reúne todo el módulo.** Determinante para el polinomio característico, espacio nulo
   para los autovectores, producto punto para verificar la ortogonalidad.
3. **La simetría lo simplifica todo.** Autovalores reales, autovectores ortogonales y
   $P^{-1} = P^\top$; por eso las matrices de covarianza son tan manejables.
4. **El módulo de los autovalores gobierna el comportamiento a largo plazo.** Es el criterio de
   estabilidad de un sistema iterado y la explicación de los gradientes que explotan o se desvanecen.

## Qué sigue

- La versión en script, más corta y sin explicaciones: [`04_autovalores_y_autovectores.py`](04_autovalores_y_autovectores.py)
- El documento de la sesión: `Matematicas_para_IA_04.pdf`
- El notebook anterior: [`03 · Determinante, norma, producto punto y ortogonalidad`](03_determinante_norma_producto_punto.ipynb)
- Los demás módulos, en el [README del repositorio](../../README.md)

**Referencia.** Deisenroth, M. P., Faisal, A. A. y Ong, C. S. (2020). *Mathematics for Machine
Learning*. Cambridge University Press, secciones 4.1–4.4.

---

*Material abierto bajo licencia MIT. ¿Encontraste un error o quieres aportar un ejercicio?
Lee [CONTRIBUTING.md](../../CONTRIBUTING.md).*